## 1. Imports e constantes

In [1]:
import csv
import json
from datetime import datetime, date

ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
LIMITE_SUSPEITO = 10000.00
FORMATO_DATA = "%Y-%m-%d"


## 2. Leitura do arquivo CSV
Lê o arquivo com o módulo `csv` nativo (sem pandas), usando `DictReader` para
acessar as colunas pelo nome. Trata `FileNotFoundError`.

In [2]:
def ler_transacoes(caminho=ARQUIVO_CSV):
    try:
        with open(caminho, newline="", encoding="utf-8") as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        print(f"ERRO: arquivo '{caminho}' não encontrado. "
              f"Verifique se ele está na mesma pasta do notebook.")
        return []


In [3]:
# Teste rápido — Parte 1
brutas = ler_transacoes()
print(f"Linhas lidas: {len(brutas)}")
print("Primeira linha bruta:", brutas[0] if brutas else "(vazio)")

Linhas lidas: 1000
Primeira linha bruta: {'id': '  ', 'data': '2026-05-23', 'cliente_id': 'CLI023', 'tipo': 'credito', 'valor': '1053.17', 'descricao': 'TED recebida', 'categoria': 'transferencia'}


## 3. Validação e limpeza
Valida cada linha. Descarta silenciosamente as inválidas. Funções pequenas e
reutilizáveis para validar data e valor isoladamente.

In [4]:
def validar_data(texto):
    try:
        return datetime.strptime(texto.strip(), FORMATO_DATA)
    except (ValueError, TypeError, AttributeError):
        return None


def validar_valor(texto):
    try:
        valor = float(texto)
    except (ValueError, TypeError):
        return None
    return valor if valor > 0 else None


def validar_transacao(linha):
    id_txt = str(linha.get("id", "")).strip()
    if not id_txt.isdigit():
        return None
    if not linha.get("cliente_id", "").strip():
        return None
    dt = validar_data(linha.get("data", ""))
    if dt is None:
        return None
    if linha.get("tipo") not in ("credito", "debito"):
        return None
    valor = validar_valor(linha.get("valor", ""))
    if valor is None:
        return None

    return {
        "id": int(id_txt),
        "data": dt,
        "mes": dt.strftime("%Y-%m"),
        "cliente_id": linha["cliente_id"].strip(),
        "tipo": linha["tipo"],
        "valor": valor,
        "descricao": linha.get("descricao", ""),
        "categoria": linha.get("categoria", ""),
    }


In [5]:
# Teste rápido — Parte 2 (resumo da limpeza)
validas, invalidas = [], 0
for ln in brutas:
    limpa = validar_transacao(ln)
    if limpa:
        validas.append(limpa)
    else:
        invalidas += 1

print(f"Total de linhas lidas: {len(brutas)}")
print(f"Linhas válidas: {len(validas)}")
print(f"Linhas inválidas: {invalidas}")
print()
print("Exemplo de registro limpo:")
print(validas[0])

Total de linhas lidas: 1000
Linhas válidas: 750
Linhas inválidas: 250

Exemplo de registro limpo:
{'id': 666, 'data': datetime.datetime(2026, 2, 10, 0, 0), 'mes': '2026-02', 'cliente_id': 'CLI033', 'tipo': 'debito', 'valor': 10582.25, 'descricao': 'Streaming', 'categoria': 'assinatura'}


## 4. Manipulação de datas
Extrai o mês (`AAAA-MM`) de cada transação e calcula o período coberto
(dias entre a transação mais antiga e a mais recente).

In [6]:
def calcular_periodo(transacoes):
    datas = [t["data"] for t in transacoes]
    mais_antiga = min(datas)
    mais_recente = max(datas)
    dias_entre = (mais_recente - mais_antiga).days
    return mais_antiga, mais_recente, dias_entre


In [7]:
# Teste rápido — Parte 3
antiga, recente, dias = calcular_periodo(validas)
print(f"Transação mais antiga: {antiga.date()}")
print(f"Transação mais recente: {recente.date()}")
print(f"Dias entre a mais antiga e a mais recente: {dias}")

Transação mais antiga: 2026-01-01
Transação mais recente: 2026-06-30
Dias entre a mais antiga e a mais recente: 180


## 5. Agrupamento mensal e métricas
Para cada mês: quantidade, total crédito, total débito, saldo, média, maior e
menor valor. Também identifica as transações suspeitas (> LIMITE_SUSPEITO).

In [8]:
def gerar_relatorio(transacoes):
    resumo = {}
    suspeitas = []

    for t in transacoes:
        mes = t["mes"]
        if mes not in resumo:
            resumo[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": t["valor"],
                "menor_valor": t["valor"],
            }
        r = resumo[mes]
        r["quantidade"] += 1
        if t["tipo"] == "credito":
            r["total_credito"] += t["valor"]
        else:
            r["total_debito"] += t["valor"]
        r["maior_valor"] = max(r["maior_valor"], t["valor"])
        r["menor_valor"] = min(r["menor_valor"], t["valor"])

        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime(FORMATO_DATA),
                "valor": t["valor"],
            })

    for r in resumo.values():
        soma = r["total_credito"] + r["total_debito"]
        r["saldo"] = round(r["total_credito"] - r["total_debito"], 2)
        r["media"] = round(soma / r["quantidade"], 2)
        for chave in ("total_credito", "total_debito", "maior_valor", "menor_valor"):
            r[chave] = round(r[chave], 2)

    resumo = {mes: resumo[mes] for mes in sorted(resumo)}
    suspeitas.sort(key=lambda s: (s["data"], s["id"]))

    return {"resumo_mensal": resumo, "suspeitas": suspeitas}


In [9]:
# Teste rápido — Parte 4
rel = gerar_relatorio(validas)
print("Meses encontrados:", list(rel["resumo_mensal"].keys()))
print()
print("Métricas de 2026-01:")
for chave, valor in rel["resumo_mensal"]["2026-01"].items():
    print(f"  {chave}: {valor}")
print()
print(f"Transações suspeitas detectadas: {len(rel['suspeitas'])}")

Meses encontrados: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

Métricas de 2026-01:
  quantidade: 116
  total_credito: 298131.28
  total_debito: 475698.9
  saldo: -177567.62
  media: 6670.95
  maior_valor: 55895.1
  menor_valor: 188.84

Transações suspeitas detectadas: 100


## 6. Exibição formatada no terminal
Relatório legível: separadores, valores em R$ no padrão brasileiro, período
analisado, totais de válidas/inválidas e lista de suspeitas.

In [10]:
def fmt_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio, total_validas, total_invalidas, periodo):
    antiga, recente, dias = periodo

    print("=" * 40)
    print("       ANÁLISE FINANCEIRA — ClearBank")
    print("=" * 40)
    print(f"Período analisado: {antiga.date()} → {recente.date()} ({dias} dias)")
    print(f"Transações válidas:   {total_validas}")
    print(f"Transações inválidas: {total_invalidas}")
    print()

    print("===== RELATÓRIO MENSAL =====")
    for mes, r in relatorio["resumo_mensal"].items():
        print()
        print(f"Mês: {mes}")
        print(f"  {'Transações:':<15}{r['quantidade']}")
        print(f"  {'Total crédito:':<15}{fmt_moeda(r['total_credito'])}")
        print(f"  {'Total débito:':<15}{fmt_moeda(r['total_debito'])}")
        print(f"  {'Saldo:':<15}{fmt_moeda(r['saldo'])}")
        print(f"  {'Média:':<15}{fmt_moeda(r['media'])}")
        print(f"  {'Maior valor:':<15}{fmt_moeda(r['maior_valor'])}")
        print(f"  {'Menor valor:':<15}{fmt_moeda(r['menor_valor'])}")

    print()
    print("===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | "
                  f"Data: {s['data']} | Valor: {fmt_moeda(s['valor'])}")


### Teste da exibição
Chama `exibir_relatorio` com os dados já processados. Como há 100 transações
acima de R$ 10.000, a seção de suspeitas é longa — isso é esperado para este
conjunto de dados.

In [11]:
# Teste rápido — Parte 5
periodo = calcular_periodo(validas)
exibir_relatorio(rel, len(validas), invalidas, periodo)

       ANÁLISE FINANCEIRA — ClearBank
Período analisado: 2026-01-01 → 2026-06-30 (180 dias)
Transações válidas:   750
Transações inválidas: 250

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:    116
  Total crédito: R$ 298.131,28
  Total débito:  R$ 475.698,90
  Saldo:         R$ -177.567,62
  Média:         R$ 6.670,95
  Maior valor:   R$ 55.895,10
  Menor valor:   R$ 188,84

Mês: 2026-02
  Transações:    122
  Total crédito: R$ 253.372,52
  Total débito:  R$ 373.854,19
  Saldo:         R$ -120.481,67
  Média:         R$ 5.141,20
  Maior valor:   R$ 54.306,20
  Menor valor:   R$ 43,80

Mês: 2026-03
  Transações:    117
  Total crédito: R$ 416.113,85
  Total débito:  R$ 438.828,09
  Saldo:         R$ -22.714,24
  Média:         R$ 7.307,20
  Maior valor:   R$ 59.433,30
  Menor valor:   R$ 136,92

Mês: 2026-04
  Transações:    119
  Total crédito: R$ 463.592,61
  Total débito:  R$ 297.780,26
  Saldo:         R$ 165.812,35
  Média:         R$ 6.398,09
  Maior valor:   R$ 57.577

## 7. Exportação do relatório em JSON
Salva em `relatorio.json` com `ensure_ascii=False, indent=2`.

In [12]:
def salvar_json(relatorio, total_validas, total_invalidas, caminho=ARQUIVO_JSON):
    dados = {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": relatorio["resumo_mensal"],
        "transacoes_suspeitas": relatorio["suspeitas"],
    }
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    print(f"Arquivo '{caminho}' gerado com sucesso.")


## Célula de Execução Principal
Orquestra todas as funções: leitura → validação → período → métricas →
relatório no terminal → JSON. Roda do início ao fim e gera o `relatorio.json`.

In [13]:
def main():
    brutas = ler_transacoes()

    validas, invalidas = [], 0
    for ln in brutas:
        limpa = validar_transacao(ln)
        if limpa:
            validas.append(limpa)
        else:
            invalidas += 1

    print(f"Total de linhas lidas: {len(brutas)}")
    print(f"Linhas válidas: {len(validas)}")
    print(f"Linhas inválidas: {invalidas}")
    print()

    periodo = calcular_periodo(validas)
    relatorio = gerar_relatorio(validas)

    exibir_relatorio(relatorio, len(validas), invalidas, periodo)
    print()

    salvar_json(relatorio, len(validas), invalidas)


main()


Total de linhas lidas: 1000
Linhas válidas: 750
Linhas inválidas: 250

       ANÁLISE FINANCEIRA — ClearBank
Período analisado: 2026-01-01 → 2026-06-30 (180 dias)
Transações válidas:   750
Transações inválidas: 250

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:    116
  Total crédito: R$ 298.131,28
  Total débito:  R$ 475.698,90
  Saldo:         R$ -177.567,62
  Média:         R$ 6.670,95
  Maior valor:   R$ 55.895,10
  Menor valor:   R$ 188,84

Mês: 2026-02
  Transações:    122
  Total crédito: R$ 253.372,52
  Total débito:  R$ 373.854,19
  Saldo:         R$ -120.481,67
  Média:         R$ 5.141,20
  Maior valor:   R$ 54.306,20
  Menor valor:   R$ 43,80

Mês: 2026-03
  Transações:    117
  Total crédito: R$ 416.113,85
  Total débito:  R$ 438.828,09
  Saldo:         R$ -22.714,24
  Média:         R$ 7.307,20
  Maior valor:   R$ 59.433,30
  Menor valor:   R$ 136,92

Mês: 2026-04
  Transações:    119
  Total crédito: R$ 463.592,61
  Total débito:  R$ 297.780,26
  Saldo:       